# Japan Fashion Items — Sample Analytics

Quick exploratory analysis on the [`otdnnc/japan-fashion-items`](https://huggingface.co/datasets/otdnnc/japan-fashion-items) dataset (~4M rows).

The dataset is loaded directly from the Hugging Face Hub. We use **streaming mode** to grab a sample (~100k rows) without downloading the full 5GB file. To analyze the entire dataset, set `STREAMING = False` below.

**Install once:**
```
pip install datasets pandas pyarrow matplotlib seaborn
```

In [ ]:
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_colwidth", 80)

REPO_ID     = "otdnnc/japan-fashion-items"
STREAMING   = True       # False = download full ~5GB
SAMPLE_SIZE = 100_000    # rows to pull when streaming

## 1. Load a sample from Hugging Face Hub

In [ ]:
if STREAMING:
    ds = load_dataset(REPO_ID, split="train", streaming=True)
    rows = list(ds.take(SAMPLE_SIZE))
    df = pd.DataFrame(rows)
    print(f"Streamed sample: {len(df):,} rows")
else:
    ds = load_dataset(REPO_ID, split="train")
    print(f"Full dataset   : {len(ds):,} rows")
    df = ds.to_pandas()

print("\nFirst record:")
print(ds[0] if not STREAMING else rows[0])
df.head(3)

## 2. Schema & memory

In [ ]:
df.info(memory_usage="deep")

## 3. Flatten nested structs

`brand`, `category`, `images`, `keywords`, `variations` are nested. Extract the fields we need for analysis.

In [ ]:
def safe_get(d, *keys):
    for k in keys:
        if d is None:
            return None
        d = d.get(k) if isinstance(d, dict) else None
    return d

df["brand_name"]    = df["brand"].apply(lambda x: safe_get(x, "name"))
df["category_name"] = df["category"].apply(lambda x: safe_get(x, "name"))
df["child_cat"]     = df["category"].apply(lambda x: safe_get(x, "child_category", "name"))
df["n_images"]      = df["images"].apply(lambda x: len(x["items"]) if x and x.get("items") is not None else 0)
df["n_colors"]      = df["variations"].apply(lambda x: len(safe_get(x, "items", "colors") or []))
df["n_sizes"]       = df["variations"].apply(lambda x: len(safe_get(x, "items", "sizes") or []))

df[["brand_name", "category_name", "child_cat", "price", "sex_name", "n_images", "n_colors", "n_sizes"]].head()

## 4. Top brands

In [ ]:
top_brands = df["brand_name"].value_counts().head(20)

ax = top_brands[::-1].plot(kind="barh", color="steelblue")
ax.set_title("Top 20 Brands by Item Count")
ax.set_xlabel("Number of items")
plt.tight_layout()
plt.show()

top_brands

## 5. Category distribution

In [ ]:
cat_counts = df["category_name"].value_counts().head(15)

ax = cat_counts.plot(kind="bar", color="coral")
ax.set_title("Top 15 Parent Categories")
ax.set_ylabel("Number of items")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Price distribution

In [ ]:
price = df["price"].dropna()
print(price.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(0))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(price.clip(upper=price.quantile(0.99)), bins=60, color="slateblue")
axes[0].set_title("Price distribution (clipped at 99th pct)")
axes[0].set_xlabel("JPY")

axes[1].hist(price[price > 0], bins=60, color="seagreen", log=True)
axes[1].set_xscale("log")
axes[1].set_title("Price distribution (log-log)")
axes[1].set_xlabel("JPY (log)")
plt.tight_layout()
plt.show()

## 7. Price by category (boxplot)

In [ ]:
top_cats = df["category_name"].value_counts().head(10).index
sub = df[df["category_name"].isin(top_cats)].copy()
sub["price_clipped"] = sub["price"].clip(upper=sub["price"].quantile(0.98))

plt.figure(figsize=(12, 5))
sns.boxplot(data=sub, x="category_name", y="price_clipped", order=top_cats, color="lightblue")
plt.xticks(rotation=45, ha="right")
plt.title("Price distribution by category (top 10)")
plt.ylabel("Price (JPY, clipped at 98th pct)")
plt.xlabel("")
plt.tight_layout()
plt.show()

## 8. Target gender (sex_name)

In [ ]:
sex_counts = df["sex_name"].value_counts(dropna=False)
print(sex_counts)

sex_counts.plot(kind="pie", autopct="%1.1f%%", figsize=(5, 5), colors=sns.color_palette("pastel"))
plt.title("Target gender share")
plt.ylabel("")
plt.show()

## 9. Images / colors / sizes per item

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ["n_images", "n_colors", "n_sizes"], ["#4c72b0", "#dd8452", "#55a868"]):
    df[col].clip(upper=df[col].quantile(0.99)).hist(ax=ax, bins=30, color=color)
    ax.set_title(f"Items per product: {col}")
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()

df[["n_images", "n_colors", "n_sizes"]].describe().round(2)

## 10. Color variations — top 20

In [ ]:
def extract_colors(v):
    colors = safe_get(v, "items", "colors") or []
    return [c.get("name") for c in colors if c and c.get("name")]

color_counts = Counter()
for cs in df["variations"].apply(extract_colors):
    color_counts.update(cs)

top_colors = pd.Series(dict(color_counts.most_common(20)))
ax = top_colors[::-1].plot(kind="barh", color="darkviolet")
ax.set_title("Top 20 Colors")
ax.set_xlabel("Occurrences")
plt.tight_layout()
plt.show()

## 11. Keyword attribute analysis

Keywords come in the format `name(attribute)` — e.g. `ナイロン(素材)`. Parse the attribute type from the parenthesis to see what kinds of tags dominate.

In [ ]:
ATTR_RE = re.compile(r"\(([^)]+)\)\s*$")

def extract_attr_types(kw):
    items = safe_get(kw, "items") or []
    out = []
    for it in items:
        name = (it or {}).get("name") or ""
        m = ATTR_RE.search(name)
        if m:
            out.append(m.group(1).strip())
    return out

attr_counts = Counter()
for ats in df["keywords"].apply(extract_attr_types):
    attr_counts.update(ats)

top_attrs = pd.Series(dict(attr_counts.most_common(20)))
ax = top_attrs[::-1].plot(kind="barh", color="teal")
ax.set_title("Top 20 keyword attribute types")
ax.set_xlabel("Occurrences")
plt.tight_layout()
plt.show()
top_attrs

## 12. Missing-value report

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
report = pd.DataFrame({"missing": missing, "pct": missing_pct}).sort_values("missing", ascending=False)
report[report["missing"] > 0]

---

### Next steps

- Run on the full dataset by setting `STREAMING = False` in cell 2 (downloads ~5GB).
- Increase `SAMPLE_SIZE` for a larger streaming sample.
- Try CLIP-style embeddings using image URLs (`images.items[*].url_500`).